<a href="https://colab.research.google.com/github/normala127/NLP_Yelp_Review_Project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Analytics Project
## Predicting Restaurant Failure: An NLP Driven Risk Assesment from Yelp Reviews

Students: Hatidza Imamovic, Asja Basovic

### 1. Dataset creation and environment setup

Three datasets are needed to create the final dataset which will be used for analysis and model training. These are:
- business.json: holds data about each restaurant
- review.json: holds all the reviews for each restaurant
- checkin.json: holds the dates of all checked in visits in a given restaurant

Each is loaded and then combined in regards to the business_id to ensure a correct join. The final output is saved as final.json.


In [1]:
!pip install pyspark
!apt-get install openjdk-11-jdk-headless -qq > /dev/null


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("project").getOrCreate()

print(spark)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
def show_shape(df):
  print((df.count(), len(df.columns)))

Loading the first dataset: business.json

In [6]:
df_business = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_business.json")
df_business.printSchema()
df_business.show()

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

Loading the second dataset: review.json

In [7]:
df_review = spark.read.json(r"/content/drive/MyDrive/yelp_academic_dataset_review.json")

In [8]:
show_shape(df_business)

(150346, 14)


In [9]:
show_shape(df_review)

(6990280, 9)


Joining df_review and df_business into joined_df

In [10]:
df_review.createOrReplaceTempView("review")
df_business.createOrReplaceTempView("business")

In [11]:
joined_df = spark.sql("""
SELECT t1.*, t2.*
FROM review t1
LEFT JOIN business t2 ON t2.business_id = t1.business_id
""")
joined_df.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------------+-----+-----+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|             address|          attributes|         business_id|          categories|        city|               hours|is_open|     latitude|  longitude|                name|postal_code|review_count|stars|state|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------

In [12]:
joined_df = joined_df.drop(df_business['business_id'])

In [13]:
show_shape(joined_df)

(6990280, 22)


Loading the third dataset: checkin.json

In [14]:
df_checkin = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_checkin.json")
df_checkin.printSchema()
df_checkin.show()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)

+--------------------+--------------------+
|         business_id|                date|
+--------------------+--------------------+
|---kPU91CF4Lq2-Wl...|2020-03-13 21:10:...|
|--0iUa4sNDFiZFrAd...|2010-09-13 21:43:...|
|--30_8IhuyMHbSOcN...|2013-06-14 23:29:...|
|--7PUidqRWpRSpXeb...|2011-02-15 17:12:...|
|--7jw19RH9JKXgFoh...|2014-04-21 20:42:...|
|--8IbOsAAxjKRoYsB...|2015-06-06 01:03:...|
|--9osgUCSDUWUkoTL...|2015-06-13 02:00:...|
|--ARBQr1WMsTWiwOK...|2014-12-12 00:44:...|
|--FWWsIwxRwuw9vIM...|2010-09-11 16:28:...|
|--FcbSxK1AoEtEAxO...|2017-08-18 19:43:...|
|--LC8cIrALInl2vyo...|2017-01-12 19:10:...|
|--MbOh2O1pATkXa7x...|2013-04-21 01:52:...|
|--N9yp3ZWqQIm7DqK...|2012-10-06 20:46:...|
|--O3ip9NpXTKD4oBS...|2010-04-17 21:07:...|
|--OS_I7dnABrXvRCC...| 2018-05-11 18:23:36|
|--S43ruInmIsGrnnk...|2010-08-29 01:17:...|
|--SJXpAa0E-GCp2sm...|2014-04-06 22:23:...|
|--Sd93OFWITqDHifM...|2013-01-09 17

In [15]:
df_checkin_new=df_checkin.withColumnRenamed('date', 'date_checkin')
df_checkin_new.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date_checkin: string (nullable = true)



Joining joined_df (review+business) and df_checkin into joined_df2

In [16]:
df_checkin_new.createOrReplaceTempView('checkin')
joined_df.createOrReplaceTempView('joined_df')

In [17]:
joined_df2 = spark.sql("""
SELECT t1.*, t2.date_checkin
FROM joined_df t1
JOIN checkin t2 ON t2.business_id = t1.business_id
""")
joined_df2.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [18]:
joined_df2.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

In [19]:
from pyspark.sql.functions import col, count

In [20]:
keywords = "Restaurant|Restaurants"
restaurant_df = joined_df2.filter(col('categories').rlike(keywords))

In [21]:
show_shape(restaurant_df)

(4715168, 23)


In [22]:
restaurant_df.groupBy('is_open').agg(count('review_id').alias('c')).show()

+-------+-------+
|is_open|      c|
+-------+-------+
|      0| 945589|
|      1|3769579|
+-------+-------+



Checking the distribution of opened and closed retaurants

In [23]:
df_isOpen=restaurant_df.filter(restaurant_df['is_open']==1)
df_isOpen.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [24]:
df_isClosed=restaurant_df.filter(restaurant_df['is_open']==0)
df_isClosed.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|       address|          attributes|          categories|        city|               hours|is_open|  latitude|  longitude|                name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|-0eUa8

Dropping a part of open restaurants based on business_id to balance out the classes

In [25]:
from pyspark.sql.functions import col, hash, count

In [26]:
id_labels = restaurant_df.select("business_id", "is_open").distinct()

majority_ids = id_labels.filter(col("is_open") == 1) \
    .withColumn("keep", (hash("business_id") % 10000) < 5)

In [27]:
minority_ids = id_labels.filter(col("is_open") == 0) \
    .withColumn("keep", (hash("business_id") % 100) < 50)

In [28]:
ids_to_keep = majority_ids.filter("keep").union(
    minority_ids.select("business_id", 'is_open', 'keep')
)

balanced_df = restaurant_df.join(ids_to_keep.select("business_id"),
                      "business_id")

In [29]:
show_shape(balanced_df)

(2849453, 23)


In [30]:
show_shape(majority_ids)

(34531, 3)


In [31]:
show_shape(minority_ids)

(16787, 3)


In [32]:
balanced_df.groupBy('is_open').agg(count('review_id').alias('c')).show()

+-------+-------+
|is_open|      c|
+-------+-------+
|      0| 945589|
|      1|1903864|
+-------+-------+



In [33]:
stratified_df = balanced_df.sampleBy('is_open', fractions={0: 0.16, 1: 0.08})

In [34]:
stratified_df.groupBy('is_open').agg(count('review_id').alias('c')).show()

+-------+------+
|is_open|     c|
+-------+------+
|      0|151958|
|      1|152441|
+-------+------+



Renaming columns and dropping uneeded columns

In [35]:
stratified_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

In [36]:
cols=stratified_df.columns
stars_columns=[i for i, c in enumerate(cols) if c=='stars']

cols[stars_columns[0]]='stars_review'
cols[stars_columns[1]]='stars_business'

stratified_df=stratified_df.toDF(*cols)

In [37]:
stratified_df.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars_review',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars_business',
 'state',
 'date_checkin']

In [38]:
final_df=stratified_df.drop(*['cool', 'funny', 'useful', 'latitude', 'longitude', 'postal_code', 'attributes', 'hours', 'categories'])

In [39]:
final_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars_review: double (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- is_open: long (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- stars_business: double (nullable = true)
 |-- state: string (nullable = true)
 |-- date_checkin: string (nullable = true)



In [40]:
show_shape(final_df)

(303993, 14)


### 2. Preprocessing

Firstly, on a global level, null and duplicate values were dropped.

This section covers:
- lowercase,
- keep only letters (from all languages) and spaces,
- remove extra spaces,
- remove private information,
- emojis, urls.

It creates a pipeline for TF-IDF and for semantic analysis as slightly different cleaning techniques are used for each.

The section also uses n-grams, and handles "not" negation effectively.


In [41]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

from pyspark.sql.functions import when, col, count, sum
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, NGram, VectorAssembler
from pyspark.ml.functions import vector_to_array

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [42]:
df = final_df.select("*")

In [43]:
df.show()

+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+------------+-------+--------------------+------------+--------------+-----+--------------------+
|         business_id|               date|           review_id|stars_review|                text|             user_id|             address|        city|is_open|                name|review_count|stars_business|state|        date_checkin|
+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+------------+-------+--------------------+------------+--------------+-----+--------------------+
|-0eUa8TsXFFy0FCxH...|2018-03-29 14:52:17|SyD_dVBBbD8OUk0W_...|         5.0|Courteous staff. ...|80oQJoVqdiqZfZeKo...|      3131 Walnut St|Philadelphia|      0|Waterfront Gourme...|          26|           4.0|   PA|2015-09-25 17:19:...|
|-0eUa8TsXFFy0FCxH...|2019-09-14 17:35:02|YNjfp5499A

In [44]:
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])

null_counts_pd = null_counts.toPandas().T
null_counts_pd.columns = ['Null Count']
null_counts_pd.index.name = 'Column'

null_counts.show()

+-----------+----+---------+------------+----+-------+-------+----+-------+----+------------+--------------+-----+------------+
|business_id|date|review_id|stars_review|text|user_id|address|city|is_open|name|review_count|stars_business|state|date_checkin|
+-----------+----+---------+------------+----+-------+-------+----+-------+----+------------+--------------+-----+------------+
|          0|   0|        0|           0|   0|      0|      0|   0|      0|   0|           0|             0|    0|           0|
+-----------+----+---------+------------+----+-------+-------+----+-------+----+------------+--------------+-----+------------+



In [45]:
df = df.dropDuplicates(['text'])
show_shape(df)

(303530, 14)


In [46]:
def clean_sentiment(df):
    return df.withColumn("text_cleaned_sentiment",
        F.trim(F.regexp_replace(
            F.regexp_replace(
                  F.regexp_replace(
                      F.regexp_replace(
                          F.lower(F.regexp_replace(F.col("text"), r'!+', ' ! ')),
                          r'\?+', ' ? '
                      ),
                      r"http\S+|www\S+|\S+@\S+", " "
                  ),
              r'[^a-z0-9\s!?_]', ' '
            ),
            r'\s+', ' '
        ))
    )

In [47]:
def clean_tfidf(df):
    return df.withColumn("text_cleaned",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.lower(F.col("text")),
                    r"http\S+|www\S+|\S+@\S+", " "
                ),
                r'[^A-Za-z\s]', " "
            )
        )
    ).withColumn("text_cleaned", F.regexp_replace(F.col("text_cleaned"), r"\s+", " "))

In [48]:
sdf = df.sampleBy('is_open', fractions={0: 0.006, 1: 0.006})
show_shape(sdf)

(1809, 14)


In [49]:
sdf=clean_tfidf(sdf)
sdf=clean_sentiment(sdf)
sdf.show(5)

+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+----------------+-------+-------------------+------------+--------------+-----+--------------------+--------------------+----------------------+
|         business_id|               date|           review_id|stars_review|                text|             user_id|             address|            city|is_open|               name|review_count|stars_business|state|        date_checkin|        text_cleaned|text_cleaned_sentiment|
+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+----------------+-------+-------------------+------------+--------------+-----+--------------------+--------------------+----------------------+
|B3kpvT4oLJdQmU9lP...|2011-01-27 03:32:19|11OcEYfvGZBICr5rJ...|         3.0|3.5. Yelp really ...|z5JZMH_CBu2XFDi1L...|          1 State St|   Santa 

Cleaning other columns (not including any IDs, review and business stars, review count and is open)

In [50]:
def clean_columns(df):
  df = df.withColumn("city_cleaned", F.trim(F.lower(F.col("city")))
          ) \
          .withColumn("state_cleaned", F.trim(F.lower(F.col("state")))
          ) \
          .withColumn("address_cleaned", F.trim(F.lower(F.regexp_replace(F.col("address"), r"\s+", " ")))
          ) \
          .withColumn("name_cleaned", F.trim(F.lower(F.regexp_replace(F.regexp_replace(F.col("name"), r"[^\p{L}\s]", ""), r"\s+", " ")))
          ) \
          .withColumn("date_parsed", F.to_date(F.col("date"), "yyyy-MM-dd HH:mm:ss")
          ) \
          .withColumn("last_checkin_date", F.to_date(F.when(F.col("date_checkin").contains(","), F.trim(F.reverse(F.split(F.col("date_checkin"), ",")).getItem(0))).otherwise(F.col('date_checkin'))))
  df=df.drop("city", "state","address", "name", "date", "date_checkin")
  return df

In [51]:
sdf2 = clean_columns(sdf)
sdf2.show(5)

+--------------------+--------------------+------------+--------------------+--------------------+-------+------------+--------------+--------------------+----------------------+----------------+-------------+--------------------+------------------+-----------+-----------------+
|         business_id|           review_id|stars_review|                text|             user_id|is_open|review_count|stars_business|        text_cleaned|text_cleaned_sentiment|    city_cleaned|state_cleaned|     address_cleaned|      name_cleaned|date_parsed|last_checkin_date|
+--------------------+--------------------+------------+--------------------+--------------------+-------+------------+--------------+--------------------+----------------------+----------------+-------------+--------------------+------------------+-----------+-----------------+
|B3kpvT4oLJdQmU9lP...|11OcEYfvGZBICr5rJ...|         3.0|3.5. Yelp really ...|z5JZMH_CBu2XFDi1L...|      0|         265|           3.5|yelp really needs...|  3 5

### 3. Feature engineering

In [52]:
from pyspark.sql.functions import month, day, year

def extract_dates(sdf):
  d = sdf.withColumn('month', F.month('date_parsed')) \
  .withColumn('month_checkin', F.month('last_checkin_date')) \
  .withColumn('day', F.day('date_parsed')) \
  .withColumn('day_checkin', F.day('last_checkin_date')) \
  .withColumn('year', F.year('date_parsed')) \
  .withColumn('year_checkin', F.year('last_checkin_date'))

  d=d.drop('date_parsed', 'last_checkin_date')
  return d


In [53]:
import numpy as np

def encode_dates(sdf):
  d = sdf.withColumn("month_sin", F.sin(2 * np.pi * F.col("month") / 12)) \
  .withColumn("month_cos", F.cos(2 * np.pi * F.col("month") / 12)) \
  .withColumn('day_sin', F.sin(2 * np.pi * F.col("day") / 7)) \
  .withColumn('day_cos', F.cos(2 * np.pi * F.col("day") / 7)) \
  .withColumn("month_checkin_sin", F.sin(2 * np.pi * F.col("month_checkin") / 12)) \
  .withColumn("month_checkin_cos", F.cos(2 * np.pi * F.col("month_checkin") / 12)) \
  .withColumn('day_checkin_sin', F.sin(2 * np.pi * F.col("day_checkin") / 7)) \
  .withColumn('day_checkin_cos', F.cos(2 * np.pi * F.col("day_checkin") / 7))

  d = d.drop('month', 'day', 'month_checkin', 'day_checkin')
  return d


In [54]:
sdf3=extract_dates(sdf2)
sdf3.show(5)

+--------------------+--------------------+------------+--------------------+--------------------+-------+------------+--------------+--------------------+----------------------+----------------+-------------+--------------------+------------------+-----+-------------+---+-----------+----+------------+
|         business_id|           review_id|stars_review|                text|             user_id|is_open|review_count|stars_business|        text_cleaned|text_cleaned_sentiment|    city_cleaned|state_cleaned|     address_cleaned|      name_cleaned|month|month_checkin|day|day_checkin|year|year_checkin|
+--------------------+--------------------+------------+--------------------+--------------------+-------+------------+--------------+--------------------+----------------------+----------------+-------------+--------------------+------------------+-----+-------------+---+-----------+----+------------+
|B3kpvT4oLJdQmU9lP...|11OcEYfvGZBICr5rJ...|         3.0|3.5. Yelp really ...|z5JZMH_CBu2

In [55]:
sdf4=encode_dates(sdf3)
sdf4.show(5)

+--------------------+--------------------+------------+--------------------+--------------------+-------+------------+--------------+--------------------+----------------------+----------------+-------------+--------------------+------------------+----+------------+-------------------+-------------------+--------------------+------------------+--------------------+-------------------+--------------------+--------------------+
|         business_id|           review_id|stars_review|                text|             user_id|is_open|review_count|stars_business|        text_cleaned|text_cleaned_sentiment|    city_cleaned|state_cleaned|     address_cleaned|      name_cleaned|year|year_checkin|          month_sin|          month_cos|             day_sin|           day_cos|   month_checkin_sin|  month_checkin_cos|     day_checkin_sin|     day_checkin_cos|
+--------------------+--------------------+------------+--------------------+--------------------+-------+------------+--------------+----

In [56]:
remover = StopWordsRemover()
default_stops = remover.getStopWords()
updated_stops = [w for w in default_stops if w != 'not']

def tfidf(vocab_size =4000, min_df = 5, suffix = ""):

  tokenizer = RegexTokenizer(
      inputCol="text_cleaned"+suffix,
      outputCol="tokens_raw"+suffix,
      pattern=r"\W+",
      gaps=True,
      toLowercase=True,
      minTokenLength=1
  )

  remover = StopWordsRemover(inputCol="tokens_raw"+suffix, outputCol="filtered_tokens"+suffix, stopWords=updated_stops)

  ngram = NGram(n=2, inputCol="filtered_tokens"+suffix, outputCol="bigrams"+suffix)

  uni_vectorizer = CountVectorizer(
        inputCol="filtered_tokens"+suffix,
        outputCol="uni_count_features"+suffix,
        vocabSize=vocab_size,
        minDF=min_df
    )

  bi_vectorizer = CountVectorizer(
      inputCol = 'bigrams'+suffix,
      outputCol = "bi_count_features"+suffix,
      vocabSize = vocab_size,
      minDF=min_df
  )

  assembler = VectorAssembler(
        inputCols=["uni_count_features"+suffix, "bi_count_features"+suffix],
        outputCol="combined_counts"+suffix
    )

  idf = IDF(inputCol="combined_counts"+suffix, outputCol="features"+suffix)

  return  [tokenizer, remover, ngram, uni_vectorizer, bi_vectorizer, assembler, idf]


In [57]:
tfidf_pipeline_stages = tfidf()
tfidf_pipeline = Pipeline(stages=tfidf_pipeline_stages)

In [58]:
sdf4 = sdf4.withColumn('stars_review_binary', when(sdf4['stars_review']<3, 0).otherwise(1))

In [59]:
from pyspark.ml.classification import LogisticRegression, NaiveBayes, RandomForestClassifier

def get_log_reg(suffix):
    return LogisticRegression(
        featuresCol=f'features{suffix}',
        labelCol='stars_review_binary',
        predictionCol=f'prediction{suffix}',
        probabilityCol=f'probability{suffix}',
        regParam=0.3,
        maxIter=10
    )

def get_naive_bayes(suffix):
    return NaiveBayes(
        featuresCol=f'features{suffix}',
        labelCol='stars_review_binary',
        predictionCol=f'prediction{suffix}',
        probabilityCol=f'probability{suffix}',
        modelType="multinomial"
    )

def get_random_forest(suffix):
    return RandomForestClassifier(
        featuresCol=f'features{suffix}',
        labelCol='stars_review_binary',
        predictionCol=f'prediction{suffix}',
        numTrees=20
    )

In [60]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

sdf4.cache()

models = [
    ("Logistic Regression", get_log_reg("_sentiment")),
    ("Naive Bayes", get_naive_bayes("_sentiment")),
    ("Random Forest", get_random_forest("_sentiment"))
]

results = []
evaluator = MulticlassClassificationEvaluator(
    labelCol="stars_review_binary",
    predictionCol="prediction_sentiment",
    metricName="accuracy"
)

for name, model in models:
    current_stages = tfidf(suffix="_sentiment")
    current_stages.append(model)

    pipeline = Pipeline(stages=current_stages)

    train_data, test_data = sdf4.randomSplit([0.8, 0.2], seed=42)

    fitted_model = pipeline.fit(train_data)
    predictions = fitted_model.transform(test_data)

    accuracy = evaluator.evaluate(predictions)
    results.append((name, accuracy))
    print(f"{name} Accuracy: {accuracy:.4f}")

print("\nFinal Results")
for name, acc in results:
    print(f"{name}: {acc}")

Logistic Regression Accuracy: 0.8320
Naive Bayes Accuracy: 0.8667
Random Forest Accuracy: 0.7680

Final Results
Logistic Regression: 0.832
Naive Bayes: 0.8666666666666667
Random Forest: 0.768


In [61]:
def sentiment_analysis(suffix="_sentiment"):
  stages = tfidf(suffix="_sentiment")

  nb = NaiveBayes(
        featuresCol=f'features{suffix}',
        labelCol='stars_review_binary',
        predictionCol=f'prediction{suffix}',
        probabilityCol=f'probability{suffix}',
        modelType="multinomial"
    )

  stages.append(nb)

  return stages

In [62]:
sentiment_analysis_pipeline_stages = sentiment_analysis()
sentiment_analysis_pipeline = Pipeline(stages=sentiment_analysis_pipeline_stages)

In [63]:
unique_bus_df = sdf4.select("business_id", "is_open").distinct()

fractions = {0: 0.8, 1: 0.8} # 80% of closed (0) and 80% of open (1)

train_ids = unique_bus_df.sampleBy("is_open", fractions, seed=42)

test_ids = unique_bus_df.join(train_ids, on="business_id", how="left_anti")

train_df = sdf4.join(train_ids.select("business_id"), on="business_id", how="inner")
test_df = sdf4.join(test_ids.select("business_id"), on="business_id", how="inner")

In [89]:
sentiment_model = sentiment_analysis_pipeline.fit(train_df)
tfidf_model = tfidf_pipeline.fit(train_df)

def prepare_features(input_df):
    df_with_sent = sentiment_model.transform(input_df)

    df_clean = df_with_sent.withColumn(
        "sentiment_score",
        vector_to_array(F.col("probability_sentiment"))[1]
    ).drop(
        "tokens_raw_sentiment", "filtered_tokens_sentiment",
        "uni_count_features_sentiment", "bi_count_features_sentiment",
        "combined_counts_sentiment", "features_sentiment"
    )

    return tfidf_model.transform(df_clean)

train_final = prepare_features(train_df)
test_final = prepare_features(test_df)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, MinMaxScaler

feature_columns=[col for col in train_final.columns if col not in ['is_open', "text", 'business_id', 'review_id', 'tokens_raw', 'filtered_tokens',
                                                                     'bigrams', 'uni_count_features', 'bi_count_features', 'combined_counts',
                                                                     'bigrams_sentiment', 'rawPrediction','stars_review', 'user_id', 'text_cleaned',
                                                                     'text_cleaned_sentiment', 'state_cleaned', 'name_cleaned', 'address_cleaned',
                                                                     'city_cleaned']]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="raw_features_scaling")

train_assembled = assembler.transform(train_final)
test_assembled = assembler.transform(test_final)

# For NB
minmax_scaler = MinMaxScaler(inputCol="raw_features_scaling", outputCol="scaled_features")
minmax_model = minmax_scaler.fit(train_final_assembled)  # fit on train only
nb_train = minmax_model.transform(train_final_assembled)
nb_test = minmax_model.transform(test_final_assembled)   # transform test with train's scaler

# For others (SVM, LogReg, XGBoost)
std_scaler = StandardScaler(inputCol="raw_features_scaling", outputCol="scaled_features", withMean=True, withStd=True)
std_model = std_scaler.fit(train_final_assembled)        # fit on train only
std_train = std_model.transform(train_final_assembled)
std_test = std_model.transform(test_final_assembled)

In [ ]:
# For NB
minmax_scaler = MinMaxScaler(inputCol="raw_features_scaling", outputCol="scaled_features")
minmax_model = minmax_scaler.fit(train_final_assembled)  # fit on train only
nb_train = minmax_model.transform(train_final_assembled)
nb_test = minmax_model.transform(test_final_assembled)   # transform test with train's scaler

# For others (SVM, LogReg, XGBoost)
std_scaler = StandardScaler(inputCol="raw_features_scaling", outputCol="scaled_features", withMean=True, withStd=True)
std_model = std_scaler.fit(train_final_assembled)        # fit on train only
std_train = std_model.transform(train_final_assembled)
std_test = std_model.transform(test_final_assembled)

### 4. Building models

In [103]:
def get_all_metrics(predictions):
    eval_multiclass = MulticlassClassificationEvaluator(labelCol="is_open", predictionCol="prediction2")
    eval_binary = BinaryClassificationEvaluator(labelCol="is_open", rawPredictionCol="rawPrediction2")

    return {
        'auc_roc': eval_binary.setMetricName('areaUnderROC').evaluate(predictions),
        'accuracy': eval_multiclass.setMetricName('accuracy').evaluate(predictions),
        'f1': eval_multiclass.setMetricName('f1').evaluate(predictions),
        'precision': eval_multiclass.setMetricName('weightedPrecision').evaluate(predictions),
        'recall': eval_multiclass.setMetricName('weightedRecall').evaluate(predictions)
    }

In [101]:
from pyspark.ml.classification import LinearSVC, NaiveBayes, LogisticRegression
from pyspark.sql import SparkSession
#from sparkxgb import XGBoostClassifier
from xgboost.spark import SparkXGBClassifier

def get_svm(featuresCol="assembled_features", labelCol="is_open"):
    return LinearSVC(
        featuresCol=featuresCol,
        labelCol=labelCol,
        predictionCol="prediction2",
        rawPredictionCol="rawPrediction2",
        maxIter=100,           # Maximum number of iterations
        regParam=0.1,          # Regularization parameter
        tol=1e-6,              # Tolerance for convergence
        standardization=True   # Standardize features
    )

def get_NB(featuresCol="assembled_features", labelCol="is_open"):
    return NaiveBayes(
        featuresCol=featuresCol,
        labelCol=labelCol,
        predictionCol="prediction2",
        rawPredictionCol="rawPrediction2",
        probabilityCol="probability2",
        smoothing=1.0,         # Laplace smoothing parameter
        modelType="multinomial"  # or "bernoulli" for binary features
    )

def get_log_reg(featuresCol="assembled_features", labelCol="is_open"):
    return LogisticRegression(
        featuresCol=featuresCol,
        labelCol=labelCol,
        predictionCol="prediction2",
        rawPredictionCol="rawPrediction2",
        probabilityCol="probability2",
        maxIter=100,           # Maximum iterations
        regParam=0.0,          # Regularization (L2 by default)
        elasticNetParam=0.0,   # 0 = L2, 1 = L1
        family="auto",         # Auto-detect binary vs multinomial
        standardization=True,
        threshold=0.5          # Binary classification threshold
    )

def get_xgb(features_col="assembled_features", label_col="is_open"):
    return SparkXGBClassifier(
        features_col=features_col,
        label_col=label_col,
        prediction_col="prediction2",
        rawPrediction_col="rawPrediction2",
        probability_col="probability2",
        n_estimators=100,      # Number of trees
        max_depth=6,           # Maximum tree depth
        learning_rate=0.3,     # Step size shrinkage
        subsample=1.0,         # Subsample ratio of training instances
        colsample_bytree=1.0,  # Subsample ratio of columns
        reg_lambda=1.0,        # L2 regularization
        reg_alpha=0.0,         # L1 regularization
        missing=float('nan'),
        seed=42
    )

In [102]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
import pandas as pd
from pyspark.ml.feature import VectorAssembler, StandardScaler, MinMaxScaler
train_final.cache()

feature_columns=[col for col in train_final.columns if col not in ['is_open', "text", 'business_id', 'review_id', 'tokens_raw', 'filtered_tokens',
                                                                     'bigrams', 'uni_count_features', 'bi_count_features', 'combined_counts',
                                                                     'bigrams_sentiment', 'rawPrediction','stars_review', 'user_id', 'text_cleaned',
                                                                     'text_cleaned_sentiment', 'state_cleaned', 'name_cleaned', 'address_cleaned',
                                                                     'city_cleaned']]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="raw_features_scaling")

train_assembled = assembler.transform(train_final)
test_assembled = assembler.transform(test_final)

# For NB
minmax_scaler = MinMaxScaler(inputCol="raw_features_scaling", outputCol="scaled_features")
minmax_model = minmax_scaler.fit(train_final_assembled)  # fit on train only
nb_train = minmax_model.transform(train_final_assembled)
nb_test = minmax_model.transform(test_final_assembled)   # transform test with train's scaler

# For others (SVM, LogReg, XGBoost)
std_scaler = StandardScaler(inputCol="raw_features_scaling", outputCol="scaled_features", withMean=True, withStd=True)
std_model = std_scaler.fit(train_final_assembled)        # fit on train only
std_train = std_model.transform(train_final_assembled)
std_test = std_model.transform(test_final_assembled)

models = [
    ('SVM', get_svm()),
    ('NB', get_NB()),
    ('LogReg', get_log_reg()),
    ('XGBoost', get_xgb())
]

results = []

for name, model in models:
    pipeline = Pipeline(stages=[assembler, model])

    try:
        fitted_model = pipeline.fit(train_final)
        predictions = fitted_model.transform(test_final)

        metrics = get_all_metrics(predictions)
        metrics['model'] = name
        results.append(metrics)
        print(f"{name} - AUC: {metrics['auc_roc']:.4f}, Accuracy: {metrics['accuracy']:.4f}")

    except Exception as e:
        print(f"Error with {name}: {str(e)}")

if results:
    results_df = pd.DataFrame(results)
    results_df = results_df[['model', 'auc_roc', 'accuracy', 'f1', 'precision', 'recall']]

    print("Model Performance Summary:")
    print(results_df.round(4).to_string(index=False))
else:
    print("No models trained successfully!")

Error with SVM: [FIELD_NOT_FOUND] No such struct field `prediction` in `business_id`, `review_id`, `stars_review`, `text`, `user_id`, `is_open`, `review_count`, `stars_business`, `text_cleaned`, `text_cleaned_sentiment`, `city_cleaned`, `state_cleaned`, `address_cleaned`, `name_cleaned`, `year`, `year_checkin`, `month_sin`, `month_cos`, `day_sin`, `day_cos`, `month_checkin_sin`, `month_checkin_cos`, `day_checkin_sin`, `day_checkin_cos`, `stars_review_binary`, `bigrams_sentiment`, `rawPrediction`, `probability_sentiment`, `prediction_sentiment`, `sentiment_score`, `tokens_raw`, `filtered_tokens`, `bigrams`, `uni_count_features`, `bi_count_features`, `combined_counts`, `features`, `assembled_features`, `rawPrediction2`, `prediction2`. SQLSTATE: 42704
Error with NB: [USER_RAISED_EXCEPTION] Vector values MUST NOT be Negative, NaN or Infinity, but got (3089,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,25,31,32,33,137,171,173,255,335,349,377,449,507,536,553,574,706,837,847,1016,2104,2925],[1880

INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.3, 'max_depth': 6, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 1.0, 'rawPrediction_col': 'rawPrediction2', 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Error with XGBoost: [FIELD_NOT_FOUND] No such struct field `prediction` in `business_id`, `review_id`, `stars_review`, `text`, `user_id`, `is_open`, `review_count`, `stars_business`, `text_cleaned`, `text_cleaned_sentiment`, `city_cleaned`, `state_cleaned`, `address_cleaned`, `name_cleaned`, `year`, `year_checkin`, `month_sin`, `month_cos`, `day_sin`, `day_cos`, `month_checkin_sin`, `month_checkin_cos`, `day_checkin_sin`, `day_checkin_cos`, `stars_review_binary`, `bigrams_sentiment`, `rawPrediction`, `probability_sentiment`, `prediction_sentiment`, `sentiment_score`, `tokens_raw`, `filtered_tokens`, `bigrams`, `uni_count_features`, `bi_count_features`, `combined_counts`, `features`, `assembled_features`, `prediction2`, `probability2`. SQLSTATE: 42704
No models trained successfully!


In [100]:
train_final.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars_review: double (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- is_open: long (nullable = true)
 |-- review_count: long (nullable = true)
 |-- stars_business: double (nullable = true)
 |-- text_cleaned: string (nullable = true)
 |-- text_cleaned_sentiment: string (nullable = true)
 |-- city_cleaned: string (nullable = true)
 |-- state_cleaned: string (nullable = true)
 |-- address_cleaned: string (nullable = true)
 |-- name_cleaned: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- year_checkin: integer (nullable = true)
 |-- month_sin: double (nullable = true)
 |-- month_cos: double (nullable = true)
 |-- day_sin: double (nullable = true)
 |-- day_cos: double (nullable = true)
 |-- month_checkin_sin: double (nullable = true)
 |-- month_checkin_cos: double (nullable = true)
 |-- day_checkin_sin: double (nullable = true)
 |

In [88]:
train_final.select('text', 'stars_review_binary', 'bigrams_sentiment', 'rawPrediction', 'prediction_sentiment', 'probability_sentiment','sentiment_score', 'tokens_raw', 'filtered_tokens', 'bigrams', 'uni_count_features', 'bi_count_features', 'combined_counts', 'features').show(5, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [104]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler, MinMaxScaler
import pandas as pd

train_final.cache()

feature_columns = [col for col in train_final.columns if col not in [
    'is_open', "text", 'business_id', 'review_id', 'tokens_raw', 'filtered_tokens',
    'bigrams', 'uni_count_features', 'bi_count_features', 'combined_counts',
    'bigrams_sentiment', 'rawPrediction', 'stars_review', 'user_id', 'text_cleaned',
    'text_cleaned_sentiment', 'state_cleaned', 'name_cleaned', 'address_cleaned', 'city_cleaned'
]]

# Step 1: assemble raw features
assembler = VectorAssembler(inputCols=feature_columns, outputCol="raw_features_scaling", handleInvalid="keep")
train_assembled = assembler.transform(train_final)
test_assembled = assembler.transform(test_final)

# Step 2a: MinMaxScaler for NB (non-negative values required)
minmax_scaler = MinMaxScaler(inputCol="raw_features_scaling", outputCol="assembled_features")
minmax_model = minmax_scaler.fit(train_assembled)       # fit on train only
nb_train = minmax_model.transform(train_assembled)
nb_test = minmax_model.transform(test_assembled)

# Step 2b: StandardScaler for SVM, LogReg, XGBoost (withMean=False for sparse vectors)
std_scaler = StandardScaler(inputCol="raw_features_scaling", outputCol="assembled_features",
                            withMean=False, withStd=True)
std_model = std_scaler.fit(train_assembled)             # fit on train only
std_train = std_model.transform(train_assembled)
std_test = std_model.transform(test_assembled)

In [105]:
models = [
    ('SVM',     get_svm()),
    ('NB',      get_NB()),
    ('LogReg',  get_log_reg()),
    ('XGBoost', get_xgb())
]

results = []

for name, model in models:
    train_data = nb_train if name == 'NB' else std_train
    test_data  = nb_test  if name == 'NB' else std_test

    pipeline = Pipeline(stages=[model])

    try:
        fitted_model = pipeline.fit(train_data)
        predictions = fitted_model.transform(test_data)

        metrics = get_all_metrics(predictions)
        metrics['model'] = name
        results.append(metrics)
        print(f"{name} - AUC: {metrics['auc_roc']:.4f}, Accuracy: {metrics['accuracy']:.4f}")

    except Exception as e:
        print(f"Error with {name}: {str(e)}")

if results:
    results_df = pd.DataFrame(results)
    results_df = results_df[['model', 'auc_roc', 'accuracy', 'f1', 'precision', 'recall']]
    print("Model Performance Summary:")
    print(results_df.round(4).to_string(index=False))
else:
    print("No models trained successfully!")

SVM - AUC: 0.7763, Accuracy: 0.7304
NB - AUC: 0.5413, Accuracy: 0.5799
LogReg - AUC: 0.7549, Accuracy: 0.7147


INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.3, 'max_depth': 6, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 1.0, 'rawPrediction_col': 'rawPrediction2', 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Error with XGBoost: [FIELD_NOT_FOUND] No such struct field `rawPrediction2` in `business_id`, `review_id`, `stars_review`, `text`, `user_id`, `is_open`, `review_count`, `stars_business`, `text_cleaned`, `text_cleaned_sentiment`, `city_cleaned`, `state_cleaned`, `address_cleaned`, `name_cleaned`, `year`, `year_checkin`, `month_sin`, `month_cos`, `day_sin`, `day_cos`, `month_checkin_sin`, `month_checkin_cos`, `day_checkin_sin`, `day_checkin_cos`, `stars_review_binary`, `bigrams_sentiment`, `rawPrediction`, `probability_sentiment`, `prediction_sentiment`, `sentiment_score`, `tokens_raw`, `filtered_tokens`, `bigrams`, `uni_count_features`, `bi_count_features`, `combined_counts`, `features`, `raw_features_scaling`, `assembled_features`, `prediction2`, `probability2`. SQLSTATE: 42704
Model Performance Summary:
 model  auc_roc  accuracy     f1  precision  recall
   SVM   0.7763    0.7304 0.7301     0.7303  0.7304
    NB   0.5413    0.5799 0.5733     0.5940  0.5799
LogReg   0.7549    0.7147 0.

to do: scaling with mean = true